# LCP Anomaly Root-Cause Report Pipeline (v2)

**Flow**: data + metadata → integrity validation → deterministic statistics → verdict selection → models (evidence only) → findings JSON → local LLM verbalization (validated) → email

Design changes vs v1:

| # | v1 problem | v2 change |
|---|---|---|
| 1 | Classifier explained *window membership*, reported as performance cause | Two honest models: window classifier = **composition fingerprint** (AUC-gated), log-timer regressor = **performance drivers** (delivery metrics included) |
| 2 | Report stats computed on sampled CSV (p75 213ms off) | Ground-truth stats from **sidecar metadata**; sample integrity gate blocks the run on drift |
| 3 | +16ms declared a "sudden worsened state" | **Severity gate**: bootstrap CI + Mann-Whitney + absolute/relative thresholds |
| 4 | No mix/outlier/behavior analysis | Deterministic modules: Oaxaca decomposition, top-mover discovery, localization check, new-visitor signal, artifact audit, delivery health |
| 5 | LLM invented "21.06%" from SHAP log-odds | LLM only **verbalizes a findings JSON**; every output number validated against the findings; deterministic fallback report if validation fails |
| 6 | Raw feature names / placeholder leaked into email | Generic feature humanization; single-pass report assembly; proper `<ul>` HTML |
| 7 | Credentials in notebook | All secrets via env / config files only |

All discovery is **generic** — no page names, countries, or campaign assumptions are hardcoded. The "traffic mix shift + cold-cache new visitors" story emerges from the rules when the data supports it; other datasets will select `delivery_regression`, `segment_regression`, or `no_action` verdicts instead.

In [ ]:
# Cell 1 — Configuration (no secrets in this notebook)
import os
from pathlib import Path

CSV_FILENAME   = os.getenv("ANOMALY_CSV", "sample_data_7-20_2357_largestcontentfulpaint_iter5_win1_inter5.csv")
PROCESSED_DIR  = Path(os.getenv("PROCESSED_DIR", "/opt/perf-analytics/processed"))
METADATA_PATH  = PROCESSED_DIR / (Path(CSV_FILENAME).stem + ".meta.json")   # sidecar from upstream

TIMER_COL, LABEL_COL = "timer", "label"
PRIMARY_DIM    = "page_group"                       # derived below; the first drilldown axis
SECONDARY_DIMS = ["country", "connectiontype", "deviceType", "isp"]
CATEGORICAL    = ["page_group","country","deviceType","os","browser","protocol",
                  "connectiontype","origin_flag","isp","landingpage","paidmedia","referrer_present"]
DELIVERY_NUMERIC = ["deviceMemory","rtt","cacherate","cdncacherate","transferbyte",
                    "bodysize","requestcount","origintime","edgetime"]

ARTIFACT_MS      = 60_000     # beacons above this are background-tab artifacts
SEVERITY_FLOOR_MS= 50         # p75 deltas below this never alert
SAMPLE_DRIFT_TOL = 3.0        # % allowed drift between sample and source stats
MIN_SEG_N        = 300        # min beacons per window for a segment to be reported
MIN_SHARE_PP     = 1.0        # min share change to qualify as a mover

OLLAMA_URL   = os.getenv("OLLAMA_URL", "http://localhost:11434/api/generate")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5:7b-instruct")
DRY_RUN_EMAIL= os.getenv("DRY_RUN_EMAIL", "1") == "1"   # default: never send by accident

# Human-label overrides for customer wording (optional, generic pipeline works without)
FEATURE_LABEL_OVERRIDES = {}
print(f"config ok — csv={CSV_FILENAME}, dry_run_email={DRY_RUN_EMAIL}")

In [ ]:
# Cell 2 — Core statistical modules
"""Core deterministic analysis functions for the LCP anomaly report pipeline.

Design principles (generic, not dataset-specific):
- No hardcoded page names, countries, or segments. All "top movers" are discovered.
- Every reported number is computed here and carried into a findings dict; the
  LLM only verbalizes findings and is validated against the allowed-number set.
"""
import re
import numpy as np
import pandas as pd
from scipy import stats

RNG = np.random.default_rng(42)

# ---------------------------------------------------------------- page groups
def derive_page_group(url_series: pd.Series, top_k: int = 15) -> pd.Series:
    """Extract a coarse page group from URL paths, generic to any site.

    Takes the first path segment after an (optional) short locale segment.
    Groups outside the top_k by traffic volume are folded into 'other'.
    """
    def _seg(u):
        if not isinstance(u, str):
            return "unknown"
        m = re.match(r"^https?://[^/]+/(.*)$", u)
        if not m:
            return "unknown"
        parts = [p for p in m.group(1).split("/") if p]
        if not parts:
            return "home"
        # treat a leading 1-5 char alnum segment as locale (uk, in, sec, latin...)
        if len(parts) >= 2 and re.fullmatch(r"[a-z_\-]{1,5}", parts[0]):
            return parts[1].split("?")[0] or "home"
        return parts[0].split("?")[0] or "home"
    seg = url_series.map(_seg)
    top = seg.value_counts().head(top_k).index
    return seg.where(seg.isin(top), "other")


# ------------------------------------------------------------- window stats
def compute_window_stats(df, timer_col="timer", label_col="label",
                         n_boot=300):
    out = {}
    for lab, key in [(0, "normal"), (1, "anomaly")]:
        s = df.loc[df[label_col] == lab, timer_col]
        out[key] = {
            "count": int(len(s)),
            "mean": round(float(s.mean()), 1),
            "p50": round(float(s.quantile(.50)), 1),
            "p75": round(float(s.quantile(.75)), 2),
            "p90": round(float(s.quantile(.90)), 1),
            "p95": round(float(s.quantile(.95)), 1),
            "p99": round(float(s.quantile(.99)), 1),
        }
    d75 = out["anomaly"]["p75"] - out["normal"]["p75"]
    out["delta_p75_ms"] = round(d75, 2)
    out["delta_p75_pct"] = round(d75 / out["normal"]["p75"] * 100, 2)

    a = df.loc[df[label_col] == 1, timer_col].to_numpy()
    n = df.loc[df[label_col] == 0, timer_col].to_numpy()
    mw = stats.mannwhitneyu(a, n, alternative="two-sided")
    out["mannwhitney_p"] = float(mw.pvalue)

    # bootstrap CI on the p75 difference
    boots = np.empty(n_boot)
    for i in range(n_boot):
        boots[i] = (np.quantile(RNG.choice(a, len(a)), .75)
                    - np.quantile(RNG.choice(n, len(n)), .75))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    out["delta_p75_ci95"] = [round(float(lo), 1), round(float(hi), 1)]
    out["delta_significant"] = bool(lo > 0 or hi < 0)
    return out


def classify_severity(win, abs_floor_ms=50):
    """Severity from the p75 delta, gated by significance and an absolute floor."""
    d, pct = win["delta_p75_ms"], win["delta_p75_pct"]
    if not win["delta_significant"] or abs(d) < abs_floor_ms:
        return "none", "No statistically meaningful change in p75."
    if pct < 0:
        return "improved", "p75 improved versus the normal window."
    if pct < 2:
        return "info", "Very small p75 increase; monitoring only."
    if pct < 5:
        return "low", "Small but significant p75 increase."
    if pct < 15:
        return "warning", "Meaningful p75 degradation."
    return "critical", "Severe p75 degradation."


# ---------------------------------------------------------------- outliers
def audit_outliers(df, timer_col="timer", label_col="label",
                   artifact_ms=60_000):
    """Quantify extreme-tail beacons (likely background-tab artifacts) and
    their influence on the mean, per window."""
    res = {"artifact_threshold_ms": artifact_ms, "windows": {}}
    for lab, key in [(0, "normal"), (1, "anomaly")]:
        s = df.loc[df[label_col] == lab, timer_col]
        art = s[s > artifact_ms]
        mean_all = float(s.mean())
        mean_clean = float(s[s <= artifact_ms].mean()) if (s <= artifact_ms).any() else np.nan
        res["windows"][key] = {
            "artifact_count": int(len(art)),
            "artifact_share_pct": round(len(art) / len(s) * 100, 3),
            "max_timer_ms": int(s.max()),
            "mean_all_ms": round(mean_all, 1),
            "mean_excl_artifacts_ms": round(mean_clean, 1),
            "mean_inflation_ms": round(mean_all - mean_clean, 1),
        }
    return res


# ------------------------------------------------------------ decomposition
def mix_within_decomposition(df, dim, timer_col="timer", label_col="label",
                             stat="mean"):
    """Oaxaca-style decomposition of the overall change in `stat` of timer
    into composition (mix) and within-segment effects along `dim`.
    `dim` may be a column name or a list of columns (joint segments)."""
    if isinstance(dim, (list, tuple)):
        key = df[list(dim)].astype(str).agg("|".join, axis=1)
        dim_name = "x".join(dim)
    else:
        key, dim_name = df[dim].astype(str), dim
    agg = "mean" if stat == "mean" else "median"
    g0 = df[df[label_col] == 0].groupby(key[df[label_col] == 0])[timer_col].agg(["count", agg])
    g1 = df[df[label_col] == 1].groupby(key[df[label_col] == 1])[timer_col].agg(["count", agg])
    j = g0.join(g1, lsuffix="0", rsuffix="1", how="outer").fillna(0)
    j["s0"] = j["count0"] / max(j["count0"].sum(), 1)
    j["s1"] = j["count1"] / max(j["count1"].sum(), 1)
    mix = float(((j.s1 - j.s0) * j[f"{agg}0"]).sum())
    within = float((j.s1 * (j[f"{agg}1"] - j[f"{agg}0"])).sum())
    total = float(df.loc[df[label_col] == 1, timer_col].agg(agg)
                  - df.loc[df[label_col] == 0, timer_col].agg(agg))
    return {"dim": dim_name, "stat": stat, "total_delta_ms": round(total, 1),
            "mix_effect_ms": round(mix, 1), "within_effect_ms": round(within, 1)}


def top_movers(df, dim, timer_col="timer", label_col="label",
               min_n=200, top=5):
    """Discover segments whose traffic share or internal p75 moved the most."""
    n0 = (df[label_col] == 0).sum()
    n1 = (df[label_col] == 1).sum()
    rows = []
    for val, sub in df.groupby(dim, dropna=False):
        c0 = (sub[label_col] == 0).sum()
        c1 = (sub[label_col] == 1).sum()
        if c0 < min_n or c1 < min_n:
            continue
        p75_0 = sub.loc[sub[label_col] == 0, timer_col].quantile(.75)
        p75_1 = sub.loc[sub[label_col] == 1, timer_col].quantile(.75)
        med_0 = sub.loc[sub[label_col] == 0, timer_col].median()
        med_1 = sub.loc[sub[label_col] == 1, timer_col].median()
        rows.append({
            "segment": str(val), "n_normal": int(c0), "n_anomaly": int(c1),
            "share_normal_pct": round(c0 / n0 * 100, 2),
            "share_anomaly_pct": round(c1 / n1 * 100, 2),
            "share_delta_pp": round(c1 / n1 * 100 - c0 / n0 * 100, 2),
            "p75_normal": round(float(p75_0), 1),
            "p75_anomaly": round(float(p75_1), 1),
            "p75_delta_ms": round(float(p75_1 - p75_0), 1),
            "median_normal": round(float(med_0), 1),
            "median_anomaly": round(float(med_1), 1),
        })
    t = pd.DataFrame(rows)
    if t.empty:
        return {"dim": dim, "share_movers": [], "perf_movers": []}
    overall_p75_normal = df.loc[df[label_col] == 0, timer_col].quantile(.75)
    t["slow_segment"] = t["p75_normal"] > overall_p75_normal
    share_movers = t.reindex(t["share_delta_pp"].abs()
                             .sort_values(ascending=False).index).head(top)
    perf_movers = t.reindex(t["p75_delta_ms"].abs()
                            .sort_values(ascending=False).index).head(top)
    return {"dim": dim,
            "share_movers": share_movers.to_dict("records"),
            "perf_movers": perf_movers.to_dict("records")}


def localization_check(df, dim, segment, timer_col="timer", label_col="label"):
    """Is the sitewide p75 change fully explained by one segment?
    Reports overall / segment-only / segment-excluded p75 shifts."""
    def p75s(d):
        return (round(float(d.loc[d[label_col] == 0, timer_col].quantile(.75)), 1),
                round(float(d.loc[d[label_col] == 1, timer_col].quantile(.75)), 1))
    inseg = df[df[dim].astype(str) == str(segment)]
    exseg = df[df[dim].astype(str) != str(segment)]
    o0, o1 = p75s(df); i0, i1 = p75s(inseg); e0, e1 = p75s(exseg)
    return {"dim": dim, "segment": str(segment),
            "overall_p75": [o0, o1], "segment_p75": [i0, i1],
            "excluded_p75": [e0, e1],
            # localized: the rest of the site did not move in the same
            # direction by more than 35% of the overall shift (signed test,
            # so an improvement outside the segment still counts as localized)
            "localized": bool((e1 - e0) < 0.35 * (o1 - o0) if (o1 - o0) > 0
                              else (e1 - e0) > 0.35 * (o1 - o0) if (o1 - o0) < 0
                              else False)}


# ------------------------------------------------------- behavioral signals
def behavior_signals(df, label_col="label",
                     landing_col="landingpage", referrer_col="referrer",
                     clientcache_col="cacherate", transfer_col="transferbyte",
                     conn_col="connectiontype", cellular_value="Cellular"):
    """Generic audience-composition signals. Flags a 'new-visitor influx'
    pattern when session-entry share rises while referrer presence and
    client cache rate fall together. Column names are parameters so the
    module ports to other beacon schemas."""
    sig = {}
    for lab, key in [(0, "normal"), (1, "anomaly")]:
        d = df[df[label_col] == lab]
        sig[key] = {
            "landing_share_pct": round(float((d[landing_col] == True).mean()) * 100, 1)
                                  if landing_col in d else None,
            "referrer_present_pct": round(float(d[referrer_col].notna().mean()) * 100, 1)
                                  if referrer_col in d else None,
            "client_cacherate_median": round(float(d[clientcache_col].median()), 1)
                                  if clientcache_col in d else None,
            "transferbyte_median": int(d[transfer_col].median())
                                  if transfer_col in d else None,
            "cellular_share_pct": round(float((d[conn_col] == cellular_value).mean()) * 100, 1)
                                  if conn_col in d else None,
        }
    n, a = sig["normal"], sig["anomaly"]
    checks = []
    if n["landing_share_pct"] is not None:
        checks.append(a["landing_share_pct"] - n["landing_share_pct"] > 2)
    if n["referrer_present_pct"] is not None:
        checks.append(n["referrer_present_pct"] - a["referrer_present_pct"] > 2)
    if n["client_cacherate_median"] not in (None, 0):
        checks.append(a["client_cacherate_median"]
                      < 0.8 * n["client_cacherate_median"])
    sig["new_visitor_influx"] = bool(checks and sum(checks) >= 2)
    return sig


# -------------------------------------------------------- delivery health
def delivery_health(df, label_col="label", tol_pct=15,
                    metrics=("edgetime", "origintime", "cdncacherate"),
                    origin_flag_col="origin_flag"):
    """CDN/origin health check: verdict is 'clean' unless a delivery metric
    degrades beyond tolerance in the anomaly window."""
    res = {"metrics": {}, "issues": []}
    for m in metrics:
        if m not in df:
            continue
        m0 = float(df.loc[df[label_col] == 0, m].median())
        m1 = float(df.loc[df[label_col] == 1, m].median())
        res["metrics"][m] = {"normal_median": round(m0, 1),
                             "anomaly_median": round(m1, 1)}
        worse = (m1 > m0 * (1 + tol_pct / 100)) if m != "cdncacherate" \
            else (m1 < m0 * (1 - tol_pct / 100))
        base_floor = 20 if m != "cdncacherate" else 0
        if worse and max(m0, m1) > base_floor:
            res["issues"].append(m)
    if origin_flag_col in df:
        o0 = float((df.loc[df[label_col] == 0, origin_flag_col] == "Y").mean()) * 100
        o1 = float((df.loc[df[label_col] == 1, origin_flag_col] == "Y").mean()) * 100
        res["metrics"]["origin_traffic_share_pct"] = {
            "normal_median": round(o0, 1), "anomaly_median": round(o1, 1)}
        if o1 > o0 + 5:
            res["issues"].append("origin_traffic_share")
    res["verdict"] = "degraded" if res["issues"] else "clean"
    return res


In [ ]:
# Cell 3 — Model modules (evidence generators, not report authors)
"""Model layer for the anomaly report pipeline.

Two models with distinct, honest roles:
1. Window classifier (context features only, NO timer, NO delivery metrics):
   answers "did the traffic composition change?" — gated by holdout AUC.
   Its SHAP output is labeled as a *composition fingerprint*, never as a
   performance cause.
2. Timer regressor on log1p(timer) (context + delivery metrics):
   answers "what drives LCP?" — the per-feature difference in mean SHAP
   between windows attributes the predicted LCP shift to features.
"""
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

MAX_CAT_CARDINALITY = 20      # top-N categories kept per column, rest folded
SHAP_SAMPLE = 15_000          # rows sampled per window for SHAP


def build_features(df, categorical_cols, numeric_cols):
    """One-hot with cardinality capping. drop_first=False so every category
    is interpretable on its own (no hidden baseline)."""
    X = pd.DataFrame(index=df.index)
    for c in numeric_cols:
        if c in df:
            X[c] = pd.to_numeric(df[c], errors="coerce")
    cats = pd.DataFrame(index=df.index)
    for c in categorical_cols:
        if c not in df:
            continue
        s = df[c].astype(str).fillna("unknown")
        top = s.value_counts().head(MAX_CAT_CARDINALITY).index
        cats[c] = s.where(s.isin(top), "other")
    if len(cats.columns):
        X = pd.concat([X, pd.get_dummies(cats, drop_first=False)], axis=1)
    return X


def window_classifier_fingerprint(df, categorical_cols, label_col="label",
                                  auc_gate=0.60, seed=42):
    """Train label(window) classifier on context features only.
    Returns AUC and, if the gate passes, the top composition-shift features."""
    X = build_features(df, categorical_cols, numeric_cols=[])
    y = df[label_col].astype(int)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25,
                                          stratify=y, random_state=seed)
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                            eval_metric="logloss", n_jobs=4,
                            random_state=seed)
    clf.fit(Xtr, ytr)
    auc = float(roc_auc_score(yte, clf.predict_proba(Xte)[:, 1]))
    result = {"holdout_auc": round(auc, 3), "gate_passed": auc >= auc_gate,
              "fingerprint": []}
    if auc < auc_gate:
        result["note"] = ("Traffic composition of the two windows is nearly "
                          "indistinguishable; composition fingerprint skipped.")
        return result
    idx = np.random.default_rng(seed).choice(
        len(Xte), min(SHAP_SAMPLE, len(Xte)), replace=False)
    sv = shap.TreeExplainer(clf).shap_values(Xte.iloc[idx])
    imp = pd.Series(np.abs(sv).mean(0), index=X.columns)
    for feat, val in imp.sort_values(ascending=False).head(8).items():
        on_share_normal = float(X.loc[y == 0, feat].mean()) * 100
        on_share_anom = float(X.loc[y == 1, feat].mean()) * 100
        result["fingerprint"].append({
            "feature": feat, "mean_abs_shap": round(float(val), 4),
            "share_normal_pct": round(on_share_normal, 2),
            "share_anomaly_pct": round(on_share_anom, 2),
            "share_delta_pp": round(on_share_anom - on_share_normal, 2)})
    return result


def timer_regressor_drivers(df, categorical_cols, numeric_cols,
                            timer_col="timer", label_col="label",
                            artifact_ms=60_000, seed=42, top=10):
    """Regress log1p(timer) on context + delivery features; attribute the
    window-to-window predicted shift via the per-feature mean-SHAP delta.
    Artifact beacons (timer > artifact_ms) are excluded from training so the
    model explains typical user experience rather than background tabs."""
    d = df[df[timer_col] <= artifact_ms].copy()
    X = build_features(d, categorical_cols, numeric_cols)
    y = np.log1p(d[timer_col].astype(float))
    reg = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08,
                           n_jobs=4, random_state=seed)
    reg.fit(X, y)
    r2 = float(reg.score(X, y))

    rng = np.random.default_rng(seed)
    parts = []
    for lab in (0, 1):
        pos = np.flatnonzero((d[label_col] == lab).to_numpy())
        parts.append(rng.choice(pos, min(SHAP_SAMPLE, len(pos)), replace=False))
    idx = np.concatenate(parts)
    Xs = X.iloc[idx]
    labs = d[label_col].to_numpy()[idx]
    sv = shap.TreeExplainer(reg).shap_values(Xs)

    mean_normal = sv[labs == 0].mean(0)
    mean_anom = sv[labs == 1].mean(0)
    delta = mean_anom - mean_normal          # log-space contribution shift
    order = np.argsort(-np.abs(delta))[:top]
    drivers = []
    for i in order:
        feat = X.columns[i]
        drivers.append({
            "feature": feat,
            "shap_delta_log": round(float(delta[i]), 4),
            "direction": "worsening" if delta[i] > 0 else "improving",
            "value_median_normal": round(float(
                pd.to_numeric(Xs.loc[labs == 0, feat], errors="coerce").median()), 2),
            "value_median_anomaly": round(float(
                pd.to_numeric(Xs.loc[labs == 1, feat], errors="coerce").median()), 2),
        })
    pred_shift_pct = round(float(np.exp(sv[labs == 1].sum(1).mean()
                                        - sv[labs == 0].sum(1).mean()) - 1) * 100, 2)
    return {"train_r2": round(r2, 3),
            "predicted_shift_pct": pred_shift_pct,
            "drivers": drivers}


In [ ]:
# Cell 4 — Findings, verdict, LLM prompt/validation, email HTML
"""Findings assembly, verdict selection, LLM verbalization and validation."""
import json
import re
import numpy as np


# ------------------------------------------------------------- humanization
def humanize_feature(feat, overrides=None, url_map=None):
    """Generic feature-name -> customer-readable phrase. Works for any
    one-hot 'col_value' name without dataset-specific hardcoding.

    When `url_map` is supplied and the feature is a page group, a concrete
    representative URL is appended so the customer knows which real pages the
    group refers to."""
    overrides = overrides or {}
    if feat in overrides:
        return overrides[feat]
    generic = {
        "page_group": "the '{v}' page section",
        "country": "traffic from region '{v}'",
        "isp": "users on network provider '{v}'",
        "connectiontype": "'{v}' network connections",
        "deviceType": "'{v}' devices",
        "os": "'{v}' devices",
        "browser": "'{v}' browser sessions",
        "protocol": "connections over '{v}'",
        "landingpage": "session-entry page views" ,
        "referrer_present": "visits arriving without a referrer",
        "paidmedia": "paid-media traffic",
        "origin_flag": "requests served via origin",
    }
    numeric = {
        "cacherate": "browser cache hit rate",
        "cdncacherate": "CDN cache hit rate",
        "transferbyte": "bytes transferred over the network",
        "bodysize": "total page weight",
        "requestcount": "number of page requests",
        "origintime": "origin response time",
        "edgetime": "edge response time",
        "rtt": "network round-trip time",
        "deviceMemory": "device memory",
    }
    if feat in numeric:
        return numeric[feat]
    for col, tpl in generic.items():
        if feat.startswith(col + "_"):
            value = feat[len(col) + 1:]
            label = tpl.format(v=value)
            if url_map and col == "page_group" and value in url_map:
                label += f" (example page: {url_map[value]})"
            return label
        if feat == col:
            return tpl.format(v="")
    return feat.replace("_", " ")


# --------------------------------------------------------- verdict selection
def select_verdict(severity, decomp_primary, localization, behavior,
                   delivery, outliers):
    """Rule-based story selection — the generic core of the report.
    Returns (verdict_code, verdict_sentence)."""
    art_norm = outliers["windows"]["normal"]["mean_inflation_ms"]
    art_anom = outliers["windows"]["anomaly"]["mean_inflation_ms"]
    outlier_note = (art_norm > 100 or art_anom > 100)

    if severity in ("none", "info", "improved"):
        s = ("No meaningful page-load degradation was found between the two "
             "windows.")
        if outlier_note:
            s += (" A small number of extreme-duration beacons inflate the "
                  "average; percentile-based views are recommended.")
        return "no_action", s

    if delivery["verdict"] == "degraded":
        return "delivery_regression", (
            "Delivery-layer metrics degraded in the anomaly window; CDN or "
            "origin behavior should be investigated first.")

    mix = decomp_primary["mix_effect_ms"]
    within = decomp_primary["within_effect_ms"]
    mix_dominant = mix > 0 and mix >= abs(within)

    if mix_dominant and localization and localization.get("localized"):
        s = ("The slowdown is a traffic-composition effect: a slower page "
             "section grew its share of traffic while the rest of the site "
             "held steady or improved.")
        if behavior.get("new_visitor_influx"):
            s += (" The incoming traffic shows a new-visitor pattern "
                  "(more session entries, fewer referrers, colder browser "
                  "caches), consistent with a campaign or event launch.")
        return "traffic_mix_shift", s
    if mix_dominant:
        return "traffic_mix_shift_broad", (
            "The slowdown is driven mainly by a shift in traffic composition "
            "across several segments rather than by pages getting slower.")
    return "segment_regression", (
        "Specific segments became genuinely slower in the anomaly window; "
        "the change is not explained by traffic composition alone.")


# ------------------------------------------------------- findings container
def collect_numbers(obj, acc=None):
    if acc is None:
        acc = set()
    if isinstance(obj, dict):
        for v in obj.values():
            collect_numbers(v, acc)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            collect_numbers(v, acc)
    elif isinstance(obj, bool):
        pass
    elif isinstance(obj, (int, float, np.integer, np.floating)):
        acc.add(round(float(obj), 2))
    return acc


def validate_report_numbers(report_text, allowed, small_int_max=15):
    """Every number in the report must exist in the findings (or be a small
    structural integer). Returns the list of unauthorized numbers."""
    bad = []
    # Representative URLs may contain digits (paths, product ids). They are
    # reference strings, not metrics, so strip URLs before number validation.
    scan_text = re.sub(r"https?://\S+", " ", report_text)
    # Include optional +/- sign so negative values (e.g., -85.1, -4.37)
    # are validated against signed numbers in findings.
    for m in re.finditer(r"(?<![\w.])[-+]?\d[\d,]*(?:\.\d+)?", scan_text):
        raw = m.group(0).replace(",", "")
        try:
            v = float(raw)
        except ValueError:
            continue
        if v <= small_int_max and v == int(v):
            continue
        cands = {round(v, 2), round(v, 1), float(int(v))}
        if not any(c in allowed for c in cands):
            # tolerate values that round-trip to an allowed number
            if not any(abs(c - a) <= 0.51 for a in allowed for c in [v]):
                bad.append(m.group(0))
    return sorted(set(bad))


# ------------------------------------------------------------ LLM prompting
def build_llm_prompt(findings_json_text):
    return f"""You are a website performance analyst writing for a business customer.

Your ONLY source of truth is the FINDINGS JSON below. Verbalize it; do not analyze.

Strict rules:
- English only.
- Use ONLY numbers that literally appear in the FINDINGS JSON. Never compute,
  convert, or invent numbers. If unsure, omit the number.
- Use the "human_label" wording for factors; never output raw feature names
  containing underscores.
- When a human_label includes an "(example page: <url>)" reference, keep that
  URL exactly as written. Never invent, shorten, or modify URLs.
- No machine-learning jargon (no SHAP, model, classifier, feature, one-hot).
- Follow the verdict: the report's storyline must match
  findings.verdict.sentence. Do not contradict it.
- If findings.delivery.verdict is "clean", explicitly reassure the customer
  that CDN and origin infrastructure show no regression.

Output format (Markdown, exactly these sections):
## Executive Summary
(2-3 sentences: the p75 transition sentence from findings.headline.transition_sentence in your own words, then the verdict.)
## What Changed
(Traffic and performance evidence: top movers, decomposition, behavior signals as applicable. When you name a page section, include its example page URL.)
## What Did Not Change
(Delivery-layer health, and segments that stayed stable or improved.)
## Recommended Actions
(3-4 practical actions matched to the verdict code: {{traffic_mix_shift: front-load optimization of the growing page section for new visitors on mobile networks; delivery_regression: CDN/origin investigation; segment_regression: per-segment performance debugging; no_action: monitoring only}}.)
## Monitoring Notes
(1-2 sentences on what to watch next.)

FINDINGS JSON:
{findings_json_text}
"""


# ------------------------------------------------------- deterministic fall-back
def render_fallback_report(f):
    """Template report used when the LLM is unavailable or fails validation
    twice. Guarantees a correct, numbers-safe report ships every time."""
    h, v = f["headline"], f["verdict"]
    lines = ["## Executive Summary", h["transition_sentence"], v["sentence"], ""]
    lines.append("## What Changed")
    for mv in f["segments"]["primary_share_movers"][:3]:
        lines.append(
            f"- {mv['human_label']}: traffic share {mv['share_normal_pct']}% -> "
            f"{mv['share_anomaly_pct']}% , p75 {mv['p75_normal']}ms -> {mv['p75_anomaly']}ms")
    b = f.get("behavior", {})
    if b.get("new_visitor_influx"):
        lines.append(
            f"- New-visitor pattern: session-entry share "
            f"{b['normal']['landing_share_pct']}% -> {b['anomaly']['landing_share_pct']}%, "
            f"browser cache hit median {b['normal']['client_cacherate_median']} -> "
            f"{b['anomaly']['client_cacherate_median']}")
    lines.append("")
    lines.append("## What Did Not Change")
    if f["delivery"]["verdict"] == "clean":
        lines.append("- CDN and origin delivery metrics show no regression.")
    if f.get("localization", {}).get("localized"):
        ex = f["localization"]["excluded_p75"]
        lines.append(f"- Excluding the focus section, sitewide p75 moved "
                     f"{ex[0]}ms -> {ex[1]}ms.")
    lines.append("")
    lines.append("## Recommended Actions")
    actions = {
        "traffic_mix_shift": [
            "Pre-optimize the growing page section's largest visual element (preload, right-sized images) for first-time mobile visitors.",
            "Apply adaptive image/video delivery for cellular connections in the growing regions.",
            "Split alerting by page group and region during campaign periods to avoid composition-driven alerts."],
        "traffic_mix_shift_broad": [
            "Review campaign traffic routing and landing-page weight across the growing segments.",
            "Split alerting by page group and region."],
        "delivery_regression": [
            "Investigate CDN cache hit rate and origin response times for the anomaly window.",
            "Check recent configuration or deployment changes on the delivery path."],
        "segment_regression": [
            "Debug the slowed segments individually (page weight, third-party tags, recent releases).",
            "Compare resource waterfalls between windows for the affected segments."],
        "no_action": [
            "No action required; continue monitoring.",
            "Consider percentile-based alerting to reduce sensitivity to extreme-duration beacons."],
    }
    for a in actions.get(v["code"], actions["no_action"]):
        lines.append(f"- {a}")
    lines.append("")
    lines.append("## Monitoring Notes")
    lines.append("Watch whether the p75 returns to baseline as the traffic composition normalizes.")
    return "\n".join(lines)


# ------------------------------------------------------------- email html
def md_to_html(text):
    lines, html, in_list = text.splitlines(), [], False
    for line in lines:
        if line.startswith("- "):
            if not in_list:
                html.append("<ul>")
                in_list = True
            html.append(f"<li>{line[2:]}</li>")
            continue
        if in_list:
            html.append("</ul>")
            in_list = False
        if line.startswith("### "):
            html.append(f"<h3>{line[4:]}</h3>")
        elif line.startswith("## "):
            html.append(f"<h2>{line[3:]}</h2>")
        elif line.startswith("# "):
            html.append(f"<h1>{line[2:]}</h1>")
        elif line.strip() == "":
            html.append("")
        else:
            line = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", line)
            html.append(f"<p>{line}</p>")
    if in_list:
        html.append("</ul>")
    return ("<html><body style='font-family:Arial,sans-serif;max-width:720px'>"
            + "\n".join(html) + "</body></html>")

In [ ]:
# Cell 5 — Data load + sample-integrity gate
import json, warnings
import pandas as pd
warnings.filterwarnings("ignore")

csv_path = PROCESSED_DIR / CSV_FILENAME
if not csv_path.is_file():
    csv_path = Path(CSV_FILENAME)          # notebook-test fallback: local file
df = pd.read_csv(csv_path)
print(f"loaded {len(df):,} rows from {csv_path}")

# ---- ground truth: prefer upstream sidecar metadata over the (possibly sampled) CSV
source_meta = None
if METADATA_PATH.is_file():
    source_meta = json.loads(METADATA_PATH.read_text())
    print("sidecar metadata found — report numbers will use SOURCE stats")
    for lab, key in [(0, "normal"), (1, "anomaly")]:
        sp = float(df.loc[df[LABEL_COL]==lab, TIMER_COL].quantile(.75))
        gt = float(source_meta["windows"][key]["p75"])
        drift = abs(sp - gt) / gt * 100
        print(f"  {key}: sample p75={sp:.1f} vs source p75={gt:.1f} (drift {drift:.2f}%)")
        if drift > SAMPLE_DRIFT_TOL:
            raise RuntimeError(
                f"Sample does not represent source data for '{key}' window "
                f"({drift:.1f}% p75 drift > {SAMPLE_DRIFT_TOL}%). "
                "Fix upstream sampling (use stratified label x timer-decile sampling) before reporting.")
else:
    print("[warn] no sidecar metadata — falling back to CSV-computed stats. "
          "Upstream should emit window counts + percentiles at export time.")

# ---- derived, dataset-agnostic features
df["page_group"] = derive_page_group(df["url"]) if "url" in df else "all"
df["referrer_present"] = df["referrer"].notna() if "referrer" in df else False

# ---- representative URL per page group, so the report can cite a concrete
#      example page instead of only an abstract group name.
def representative_urls(frame, group_col="page_group", url_col="url",
                        skip=("other", "unknown", "all")):
    """Most frequent concrete URL (query string stripped) per page group."""
    mapping = {}
    if url_col not in frame:
        return mapping
    for grp, sub in frame.groupby(group_col):
        if str(grp) in skip:
            continue
        urls = sub[url_col].dropna().astype(str).map(lambda u: u.split("?")[0])
        if urls.empty:
            continue
        mapping[str(grp)] = urls.value_counts().index[0]
    return mapping

PAGE_GROUP_URLS = representative_urls(df)
print(f"representative URLs mapped for {len(PAGE_GROUP_URLS)} page groups")

In [ ]:
# Cell 6 — Step A: window statistics + severity gate
win = compute_window_stats(df, TIMER_COL, LABEL_COL)
if source_meta:   # override headline percentiles with source ground truth
    for key in ("normal", "anomaly"):
        win[key].update({k: source_meta["windows"][key][k]
                         for k in ("count","mean","p50","p75","p90","p95","p99")
                         if k in source_meta["windows"][key]})
    win["delta_p75_ms"]  = round(win["anomaly"]["p75"] - win["normal"]["p75"], 2)
    win["delta_p75_pct"] = round(win["delta_p75_ms"] / win["normal"]["p75"] * 100, 2)

severity, severity_msg = classify_severity(win, abs_floor_ms=SEVERITY_FLOOR_MS)
print(f"p75: {win['normal']['p75']} -> {win['anomaly']['p75']} "
      f"({win['delta_p75_ms']:+}ms, {win['delta_p75_pct']:+}%)  "
      f"CI95={win['delta_p75_ci95']}  MW-p={win['mannwhitney_p']:.2e}")
print(f"severity: {severity} — {severity_msg}")

In [ ]:
# Cell 7 — Step B: artifact/outlier audit
outliers = audit_outliers(df, TIMER_COL, LABEL_COL, ARTIFACT_MS)
for k, v in outliers["windows"].items():
    print(f"{k:>7}: {v['artifact_count']} beacons >{ARTIFACT_MS/1000:.0f}s "
          f"({v['artifact_share_pct']}%), mean inflated by {v['mean_inflation_ms']}ms, "
          f"max={v['max_timer_ms']:,}ms")

In [ ]:
# Cell 8 — Step C: decomposition, mover discovery, auto drilldown
decomp = {}
for d in [PRIMARY_DIM, *SECONDARY_DIMS[:2], [PRIMARY_DIM, SECONDARY_DIMS[0]]]:
    r = mix_within_decomposition(df, d, TIMER_COL, LABEL_COL)
    decomp[r["dim"]] = r
    print(f"[{r['dim']:<22}] Δmean={r['total_delta_ms']:+}ms = "
          f"mix {r['mix_effect_ms']:+} + within {r['within_effect_ms']:+}")

primary_movers = top_movers(df, PRIMARY_DIM, TIMER_COL, LABEL_COL, min_n=MIN_SEG_N)

# ---- generic focus selection: slow segments that gained meaningful share,
#      else segments whose own p75 degraded the most
focus, focus_reason = None, None
share_cands = [r for r in primary_movers["share_movers"]
               if r["share_delta_pp"] > MIN_SHARE_PP and r["slow_segment"]]
if share_cands:
    focus, focus_reason = share_cands[0], "slow segment gained traffic share"
else:
    perf_cands = [r for r in primary_movers["perf_movers"]
                  if r["p75_delta_ms"] > SEVERITY_FLOOR_MS and r["share_anomaly_pct"] > 3]
    if perf_cands:
        focus, focus_reason = perf_cands[0], "segment's own p75 degraded"

localization, drilldown = None, {}
focus_df = df
if focus:
    localization = localization_check(df, PRIMARY_DIM, focus["segment"], TIMER_COL, LABEL_COL)
    focus_df = df[df[PRIMARY_DIM].astype(str) == focus["segment"]]
    for d in SECONDARY_DIMS:
        drilldown[d] = top_movers(focus_df, d, TIMER_COL, LABEL_COL, min_n=max(100, MIN_SEG_N // 2))
    print(f"\nfocus: {PRIMARY_DIM}='{focus['segment']}' ({focus_reason}); "
          f"localized={localization['localized']}")
    top_drill = drilldown[SECONDARY_DIMS[0]]["share_movers"][:3]
    for r in top_drill:
        print(f"   {SECONDARY_DIMS[0]}={r['segment']}: share {r['share_normal_pct']}% -> "
              f"{r['share_anomaly_pct']}%  median {r['median_normal']} -> {r['median_anomaly']}ms")
else:
    print("no focus segment found — degradation is diffuse")

In [ ]:
# Cell 9 — Step D+E: audience behavior signals + delivery-layer health
behavior_overall = behavior_signals(df, LABEL_COL)
behavior_focus   = behavior_signals(focus_df, LABEL_COL)
print("new-visitor influx — overall:", behavior_overall["new_visitor_influx"],
      "| focus segment:", behavior_focus["new_visitor_influx"])

delivery = delivery_health(df, LABEL_COL)
print("delivery verdict:", delivery["verdict"], delivery["issues"] or "")
for m, v in delivery["metrics"].items():
    print(f"   {m}: {v['normal_median']} -> {v['anomaly_median']}")

In [ ]:
# Cell 10 — Step F: models as evidence (skipped when nothing to explain)
fingerprint, drivers = None, None
if severity not in ("none", "improved"):
    fingerprint = window_classifier_fingerprint(df, CATEGORICAL, LABEL_COL)
    print(f"composition fingerprint — holdout AUC={fingerprint['holdout_auc']} "
          f"(gate {'passed' if fingerprint['gate_passed'] else 'FAILED — windows statistically similar'})")
    for f_ in fingerprint["fingerprint"][:6]:
        print(f"   {f_['feature']:<38} share {f_['share_normal_pct']}% -> "
              f"{f_['share_anomaly_pct']}%")
    drivers = timer_regressor_drivers(df, CATEGORICAL, DELIVERY_NUMERIC,
                                      TIMER_COL, LABEL_COL, ARTIFACT_MS)
    print(f"\nLCP drivers — regressor R2={drivers['train_r2']}, "
          f"predicted window shift={drivers['predicted_shift_pct']}%")
    for d_ in drivers["drivers"][:8]:
        print(f"   {d_['feature']:<38} {d_['direction']:<9} "
              f"median {d_['value_median_normal']} -> {d_['value_median_anomaly']}")
else:
    print("severity gate: models skipped")

In [ ]:
# Cell 11 — Step G: findings JSON + verdict (single source of truth for the report)
from datetime import datetime, timezone

verdict_code, verdict_sentence = select_verdict(
    severity, decomp[PRIMARY_DIM], localization, behavior_focus, delivery, outliers)

def _label_movers(movers, dim):
    out = []
    for r in movers:
        r = dict(r)
        r["human_label"] = humanize_feature(
            f"{dim}_{r['segment']}", FEATURE_LABEL_OVERRIDES, url_map=PAGE_GROUP_URLS)
        out.append(r)
    return out

timer_name = "largestcontentfulpaint"
findings = {
    "meta": {"metric": timer_name, "rows": int(len(df)),
             "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
             "stats_source": "sidecar_metadata" if source_meta else "csv_sample"},
    "headline": {
        "p75_normal": win["normal"]["p75"], "p75_anomaly": win["anomaly"]["p75"],
        "delta_ms": win["delta_p75_ms"], "delta_pct": win["delta_p75_pct"],
        "ci95": win["delta_p75_ci95"], "severity": severity,
        "transition_sentence": (
            f"{timer_name} p75 moved from {win['normal']['p75']}ms (normal window) to "
            f"{win['anomaly']['p75']}ms (anomaly window), a change of "
            f"{win['delta_p75_ms']}ms ({win['delta_p75_pct']}%).")},
    "verdict": {"code": verdict_code, "sentence": verdict_sentence},
    "segments": {
        "primary_dim": PRIMARY_DIM,
        "primary_share_movers": _label_movers(primary_movers["share_movers"][:5], PRIMARY_DIM),
        "focus": (dict(focus, human_label=humanize_feature(
            f"{PRIMARY_DIM}_{focus['segment']}", FEATURE_LABEL_OVERRIDES, url_map=PAGE_GROUP_URLS),
            reason=focus_reason) if focus else None),
        "drilldown": {d: _label_movers(t["share_movers"][:4], d)
                      for d, t in drilldown.items()}},
    "localization": localization,
    "behavior": behavior_focus | {"scope": (focus["segment"] if focus else "overall")},
    "delivery": delivery,
    "outliers": outliers,
    "decomposition": decomp[PRIMARY_DIM],
    "composition_fingerprint": ([{**f_, "human_label": humanize_feature(
        f_["feature"], FEATURE_LABEL_OVERRIDES, url_map=PAGE_GROUP_URLS)} for f_ in fingerprint["fingerprint"][:6]]
        if fingerprint and fingerprint["gate_passed"] else []),
    "performance_drivers": ([{**d_, "human_label": humanize_feature(
        d_["feature"], FEATURE_LABEL_OVERRIDES, url_map=PAGE_GROUP_URLS)} for d_ in drivers["drivers"][:6]]
        if drivers else []),
}
allowed_numbers = collect_numbers(findings)
findings_text = json.dumps(findings, indent=2, ensure_ascii=False, default=str)
print(f"verdict={verdict_code} | severity={severity} | "
      f"{len(allowed_numbers)} numbers whitelisted for LLM validation")

In [ ]:
# Cell 12 — Step H: LLM verbalization (used as-is, with label/URL auto-enrichment)
import os
import re
import requests

def call_ollama(prompt, url=OLLAMA_URL, model=OLLAMA_MODEL, timeout=300):
    r = requests.post(url, json={"model": model, "prompt": prompt,
                                 "stream": False, "options": {"temperature": 0}},
                      timeout=timeout)
    r.raise_for_status()
    return r.json().get("response", "")

def enrich_report_labels(text, findings, url_map=None):
    """Post-process an LLM draft so page sections always appear with their
    human label and representative URL, even when the model emitted a raw
    feature name (e.g. page_group_unpacked) or dropped the URL."""
    url_map = url_map or {}

    # 1) collect raw one-hot feature name -> human label pairs from findings
    feat_to_label = {}
    seg = findings.get("segments", {})
    primary_dim = seg.get("primary_dim", "")
    for r in seg.get("primary_share_movers", []):
        feat_to_label[f"{primary_dim}_{r['segment']}"] = r["human_label"]
    focus = seg.get("focus")
    if focus:
        feat_to_label[f"{primary_dim}_{focus['segment']}"] = focus["human_label"]
    for dim, rows in (seg.get("drilldown") or {}).items():
        for r in rows:
            feat_to_label[f"{dim}_{r['segment']}"] = r["human_label"]
    for r in findings.get("composition_fingerprint", []):
        feat_to_label[r["feature"]] = r["human_label"]
    for r in findings.get("performance_drivers", []):
        feat_to_label[r["feature"]] = r["human_label"]

    # replace longest feature names first so shorter names can't partial-match
    for feat in sorted(feat_to_label, key=len, reverse=True):
        text = re.sub(r"(?<!\w)" + re.escape(feat) + r"(?!\w)",
                      feat_to_label[feat], text)

    # 2) ensure every page-group mention carries its representative URL
    for v, url in url_map.items():
        base = f"the '{v}' page section"
        text = re.sub(re.escape(base) + r"(?!\s*\(example page:)",
                      f"{base} (example page: {url})", text)
    return text

prompt = build_llm_prompt(findings_text)
report_md, report_source = None, "fallback_template"
try:
    report_md = call_ollama(prompt)
    report_md = enrich_report_labels(report_md, findings, PAGE_GROUP_URLS)
    report_source = f"llm+enriched ({OLLAMA_MODEL})"
    # Advisory only: we no longer reject the draft. Surface anything still off
    # after enrichment, but the (enriched) LLM output is used as the report.
    bad = validate_report_numbers(report_md, allowed_numbers)
    raw_feat = [f_["feature"] for f_ in findings["performance_drivers"]
                if f_["feature"] in report_md]
    if bad or raw_feat:
        print(f"[advisory] LLM report used as-is despite: "
              f"unauthorized numbers={bad[:5]}, raw feature names={raw_feat[:3]}")
except Exception as e:
    print(f"[warn] LLM unavailable ({e}); using deterministic fallback")
    report_md = render_fallback_report(findings)

print(f"report source: {report_source}\n")
print(report_md)

In [ ]:
# Cell 13 — Step I: email assembly + send (env-configured, DRY_RUN disabled by request)
DRY_RUN_EMAIL = False

email_subject = (f"[{severity.upper()}] {timer_name} p75 "
                 f"{win['normal']['p75']} -> {win['anomaly']['p75']}ms "
                 f"({win['delta_p75_pct']:+}%) — {verdict_code.replace('_',' ')}")
email_plain = "\n".join([f"# {timer_name} anomaly report", "",
                          findings["headline"]["transition_sentence"], "",
                          report_md])
email_html = md_to_html(email_plain)
print("subject:", email_subject)

if DRY_RUN_EMAIL:
    Path("report_preview.html").write_text(email_html)
    print("DRY RUN — email not sent; preview written to report_preview.html")
else:
    import boto3
    # secrets stay OUTSIDE the notebook: files readable only by the service account
    def _load_conf(path):
        conf = {}
        for raw in Path(path).read_text().splitlines():
            line = raw.strip()
            if line and not line.startswith("#") and "=" in line:
                k, _, v = line.partition("=")
                conf[k.strip()] = v.strip()
        return conf

    sec = _load_conf("/opt/perf-analytics/.sec/aws-ses")
    mail = _load_conf("/opt/perf-analytics/config/ses_email.conf")
    to_addresses = ["hyoon@akamai.com"]
    ses = boto3.client(
        "sesv2",
        region_name=mail.get("SES_REGION", "ap-northeast-1"),
        aws_access_key_id=sec["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=sec["AWS_SECRET_ACCESS_KEY"],
    )
    resp = ses.send_email(
        FromEmailAddress=mail["SES_FROM_EMAIL"],
        Destination={"ToAddresses": to_addresses},
        Content={
            "Simple": {
                "Subject": {"Data": email_subject, "Charset": "UTF-8"},
                "Body": {
                    "Text": {"Data": email_plain, "Charset": "UTF-8"},
                    "Html": {"Data": email_html, "Charset": "UTF-8"},
                },
            }
        },
    )
    print("sent:", resp["MessageId"])
    print("to:", ", ".join(to_addresses))

## Upstream requirements (outside this notebook)

1. **Sidecar metadata** — the CSV exporter must also write `<csv_stem>.meta.json`:
```json
{"windows": {"normal":  {"count": 0, "mean": 0, "p50": 0, "p75": 0, "p90": 0, "p95": 0, "p99": 0},
             "anomaly": {"count": 0, "mean": 0, "p50": 0, "p75": 0, "p90": 0, "p95": 0, "p99": 0}},
 "sampling": {"method": "stratified", "strata": "label x timer_decile", "rate": 1.0, "seed": 42},
 "window_boundaries": {"normal": ["...","..."], "anomaly": ["...","..."]}}
```
2. **Stratified sampling** — if sampling is needed, stratify by `label × timer decile` so percentiles survive; the Cell-5 gate blocks the run otherwise.
3. **Timestamp column** — add beacon timestamps to the export to enable window verification and minute-level trend analysis.
4. **Credential hygiene** — the FTP password that appeared in v1 must be rotated; all credentials live in `/opt/perf-analytics/.sec/` with restricted permissions.